In [ ]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from cleaning_utils import handle_numerical_missing_data, handle_categorical_missing_data

FILE_PATH = Path("../data/raw/spaceship titanic.csv")
df = pd.read_csv(FILE_PATH)
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [96]:
df.shape

(8693, 14)

In [97]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 3.4 MB


In [98]:
# check missing values
df.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [99]:
# fix cryo sleep data type
# fill first otheriwse NaN value convert to True
df.CryoSleep = df.CryoSleep.fillna(False).astype(bool)

In [100]:
# handle name column (split it into surname first)
df["Surname"] = df.Name.str.split(" ").str[-1]
# fill missing values with Unknown
df.Surname = df.Surname.fillna("Unknown")
df.drop("Name", inplace=True, axis=1)

# use Unknown to fill Cabin column
df.Cabin = df.Cabin.fillna("Unknown")

In [ ]:
# use median to fill numerical features so that we don't lose data
numeric_cols = df.select_dtypes(include="number").columns

for numeric_col in numeric_cols:
    df[numeric_col] = handle_numerical_missing_data(df[numeric_col])

In [102]:
# use mode for categorical columns
cat_cols = ["HomePlanet", "VIP", "Destination",]

for cat_col in cat_cols:
    df[cat_col] = handle_categorical_missing_data(df[cat_col])

In [103]:
# check final result
df.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
Surname         0
dtype: int64

In [104]:
# move Trasported column to end
col_to_end = df.pop("Transported")
df.insert(len(df.columns), "Transported", col_to_end)

In [105]:
# check final df
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Surname,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Santantines,True


In [106]:
# save dataset
df.to_csv("../data/processed/spaceship_titanic_cleaned.csv", index=False)